In [23]:
# ✅ Install required dependencies (run once per environment)
%pip install --upgrade pip
%pip install ultralytics opencv-python pandas matplotlib tqdm

# ✅ RTX 5070 support (sm_120) — PyTorch nightly CUDA 12.8+
%pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, torch
print("exe:", sys.executable) 
print("torch:", torch.__version__) #print torch version
print("cuda_available:", torch.cuda.is_available())#check if cuda available
print("cuda_version:", torch.version.cuda)#print version of cuda

exe: c:\Users\these\Desktop\FYP2025-SeanMaloney\venv\Scripts\python.exe
torch: 2.11.0.dev20260203+cu128
cuda_available: True
cuda_version: 12.8


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected") #checks to see if running off gpu


CUDA available: True
GPU name: NVIDIA GeForce RTX 5070 Laptop GPU


In [4]:
import os
import cv2
import shutil
import random
import zipfile
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\these\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Future use not needed right now (below)

In [5]:
# %%
import shutil #imports shutil to provide file and directory operations
from pathlib import Path #imports path from pathlib and allows filesystem paths to be handled

# Paths to clean
paths_to_clean = [
    "datasets/VisDrone",                                #dataset directory
    "runs/train/visdrone_parking_detector_vehicle_only",#training run 
    "runs/train/visdrone_parking_detector",
    "runs/train/visdrone_full_dataset_retrain",
    "runs/detect",
    "runs/val"
]

for p in paths_to_clean:
    path = Path(p)
    if path.exists():
        try:
            shutil.rmtree(path, ignore_errors=True)
            print(f"🧹 Force deleted: {p}")
        except Exception as e:
            print(f"⚠️ Could not delete {p}: {e}")
    else:
        print(f"✅ Already clean: {p}")

print("\n✨ Dataset and YOLO output folders have been fully reset. Ready for a fresh setup!")


✅ Already clean: datasets/VisDrone
✅ Already clean: runs/train/visdrone_parking_detector_vehicle_only
✅ Already clean: runs/train/visdrone_parking_detector
✅ Already clean: runs/train/visdrone_full_dataset_retrain
✅ Already clean: runs/detect
✅ Already clean: runs/val

✨ Dataset and YOLO output folders have been fully reset. Ready for a fresh setup!


In [35]:
# %%
import shutil
from pathlib import Path

# Specific subfolders that block re-download
paths = [
    "datasets/VisDrone/VisDrone2019-DET-train",     #Training dataset folder
    "datasets/VisDrone/VisDrone2019-DET-val",       #Validation dataset folder
    "datasets/VisDrone/VisDrone2019-DET-test-dev",  #Test dataset folder
]
#Loop through each directory in the list
for p in paths:
    path = Path(p)  #convert string path into a path object for safer file handling
    if path.exists():   #check if directory exists on disk
        shutil.rmtree(path, ignore_errors=True) #ignore errors, delete the directory and all of it's contents, some file are locked or already removed
        print(f"🧹 Deleted: {p}") #print confirm if directory was deleted
    else:
        print(f"✅ Already clean: {p}") #if there nothing to remove inform

print("\n✨ Old partial VisDrone folders removed. Ready to download fresh.")


✅ Already clean: datasets/VisDrone/VisDrone2019-DET-train
✅ Already clean: datasets/VisDrone/VisDrone2019-DET-val
✅ Already clean: datasets/VisDrone/VisDrone2019-DET-test-dev

✨ Old partial VisDrone folders removed. Ready to download fresh.


In [7]:
# %%
import os
from pathlib import Path
from ultralytics.utils.downloads import download #imports ultralitics built in download utility which downloads and extracts dataset safely and efficiently

# === STEP 1: Recreate core folder structure ===
base_dirs = [
    "datasets",         #root folder for datasets
    "datasets/VisDrone",#Folder specifically for Visdrone dataset
    "runs",             #Root folder for YOLO outputs
    "runs/detect",      #Folder for detection outputs
    "runs/train",       #Folder for training runs
    "runs/val",         #Folder for validation results
]

for d in base_dirs: #loop through each directory in the list
    Path(d).mkdir(parents=True, exist_ok=True) #create directory and any missing parent directories, exist_ok prevents errors if the folder already exists
    print(f"📁 Created: {d}") #print conformation

# === STEP 2: Download VisDrone dataset ===
dataset_dir = Path("datasets/VisDrone") #define the base directory where visdrone dataset will be stored
urls = [
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip", #url for cisdfrone training dataset
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",   #url for cisdrone validation dataset
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip",#url for visdrone test dataset
]

print("\n⬇️ Downloading VisDrone dataset... (this may take a while)") #informs usee that dataset download is starting
download(urls, dir=dataset_dir, threads=4)                             #downloads and extracts dataset zip file into specified directory using 4 parrell threads
print("\n✅ VisDrone dataset downloaded successfully!")                #informs user of successful download

# === STEP 3: Verify ===
expected_dirs = [
    "VisDrone2019-DET-train",   #expected training folder
    "VisDrone2019-DET-val",     #expected validation folder
    "VisDrone2019-DET-test-dev" #expected test folder
]
print("\n📦 Verifying extraction...")   #indicates the dtart of datset verification
for d in expected_dirs:
    path = dataset_dir / d #^... constructs the full path to expected dataset directory
    if path.exists():   #checks if directory exists
        print(f"✅ Found: {path}")  #confirms successful extraction
    else:   
        print(f"❌ Missing: {path} (check if extraction failed)") #warn the user that something went wrong

print("\n✨ Dataset structure rebuilt, VisDrone ready for YOLO conversion!")


📁 Created: datasets
📁 Created: datasets/VisDrone
📁 Created: runs
📁 Created: runs/detect
📁 Created: runs/train
📁 Created: runs/val

⬇️ Downloading VisDrone dataset... (this may take a while)
Unzipping datasets\VisDrone\VisDrone2019-DET-val.zip to C:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\datasets\VisDrone\VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 2.1Kfiles/s 0.5s0.0s0.6s<2.4ss
Unzipping datasets\VisDrone\VisDrone2019-DET-test-dev.zip to C:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\datasets\VisDrone\VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.8Kfiles/s 1.7s0.0s
Unzipping datasets\VisDrone\VisDrone2019-DET-train.zip to C:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\datasets\VisDrone\VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 2.1Kfiles/s 6.2s<0.0s

✅ VisDrone dataset downloaded successfully!

📦 Verifying extraction...
✅ Found: datasets\VisDrone\VisDrone2019-DET-train
✅ Found: datasets\VisDrone\VisDrone2019-DET-val
✅ Found: datasets\V

In [8]:
# ✅ Safe VisDrone → YOLO conversion (re-run friendly)
import shutil
from ultralytics.utils import TQDM
from PIL import Image #read image dimensions (height and width)

def visdrone2yolo(dir, split, source_name=None): #dir: base of directory, #split: dataset split name(train/val/test), Sourcename: original visdrone folder name
    source_dir = dir / (source_name or f"VisDrone2019-DET-{split}") #determines the source directory to convert

    # Skip if source folder doesn't exist
    if not source_dir.exists():
        print(f"⚠️ Skipping {source_name}: folder not found")
        return #stops execution for this split if the folder is missing

    # Output folders
    images_dir = dir / "images" / split
    labels_dir = dir / "labels" / split
    #defines yolo style output directories for images and labels
    labels_dir.mkdir(parents=True, exist_ok=True)
    images_dir.mkdir(parents=True, exist_ok=True)
    #create output directories if they do not already exist

    # Move images
    source_images_dir = source_dir / "images"       #path to original cisdrone images
    if source_images_dir.exists():                  #check that image folder exists
        for img in source_images_dir.glob("*.jpg"): #loop through all jpg images
            img.rename(images_dir / img.name)       #move each image into yolo images/<split> folder

    # Convert labels
    annotations = source_dir / "annotations"        #patch to visdrone annotation text files
    if not annotations.exists():                    #if annotations folder does not exists
        print(f"⚠️ No annotations found in {source_name}, skipping.")
        return  #skip conversion for this split

    for f in TQDM(annotations.glob("*.txt"), desc=f"Converting {split}"):   #loop through each annotation files with a progress bar
        img_file = images_dir / f.with_suffix(".jpg").name                  #construct the corresponding image filename
        if not img_file.exists():                                           #if the image for this annotation is missing
            print(f"⚠️ Missing image for annotation: {img_file.name}")
            continue                                                        #skip this annotation file

        img_size = Image.open(img_file).size                                #open the image and retrieve it's witdth and height
        dw, dh = 1.0 / img_size[0], 1.0 / img_size[1]                       #compute normalization factors for YOLO format
        lines = []                                                          #prepare a list to store YOLO label lines

        with open(f, encoding="utf-8") as file:                             #open visdrone annotation file
            for row in [x.split(",") for x in file.read().strip().splitlines()]: #read each annotatrion row and split by commas
                if row[4] != "0":   # ignore ignored regions
                    x, y, w, h = map(int, row[:4])                          #extract bounding box co ordinates (top-left x, y, width, height)
                    cls = int(row[5]) - 1                                   #covert visdrone class index to YOLO class index (0-based)
                    x_center, y_center = (x + w / 2) * dw, (y + h / 2) * dh #calculate normalized center co ordinates
                    w_norm, h_norm = w * dw, h * dh                         #normalize width and height
                    lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n") #append formatted YOLO annotation line

        (labels_dir / f.name).write_text("".join(lines), encoding="utf-8")  #write all YOLO annotations for this image to a label file


# ================================
#       SAFE MAIN LOOP
# ================================

#define a mapping of original visdrone folders to YOLO splits
splits = {
    "VisDrone2019-DET-train": "train",
    "VisDrone2019-DET-val": "val",
    "VisDrone2019-DET-test-dev": "test"
}

for folder, split in splits.items():                        #Loop through each visdrone dataset split
    visdrone2yolo(dataset_dir, split, folder)               #convert current visdrone split into YOLO format

    # ✅ Delete only if exists
    folder_path = dataset_dir / folder                      #construct path to orginal video
    if folder_path.exists():                                #check if original folder still exists
        shutil.rmtree(folder_path)                          #delete original Visdrone folder to avoid duplication
        print(f"🗑️ Deleted {folder}")
    else:
        print(f"⚠️ {folder} already deleted, skipping")     

print("✅ Dataset converted to YOLO format successfully.")


Converting train: ━━━━━━━━━━━━ 6471 72.7it/s 1:18s
🗑️ Deleted VisDrone2019-DET-train
Converting val: ━━━━━━━━━━━━ 548 95.1it/s 6.6s
🗑️ Deleted VisDrone2019-DET-val
Converting test: ━━━━━━━━━━━━ 1610 89.1it/s 19.1s
🗑️ Deleted VisDrone2019-DET-test-dev
✅ Dataset converted to YOLO format successfully.


In [9]:
yaml_path = dataset_dir / "data.yaml" #creates a filepath to data.yaml inside dataset directory, tells yolo where the dataset is and what classes exist

#oepn or create data.yaml starts with a multi line string to do it the f allows for cariables to be inserted

#{dataset_dir} specifies the root directory of the dataset
#yolo uses this as the base path for all relative folders
yaml_path.write_text(f"""             
path: {dataset_dir}

train: images/train
val: images/val
test: images/test

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
""")

print("✅ data.yaml created at:", yaml_path)


✅ data.yaml created at: datasets\VisDrone\data.yaml


In [10]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

#double recheck if cuda is availivle, check version and double double check it's using the gpu instead of cpu

Torch version: 2.11.0.dev20260203+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [11]:
# %%
import os
import torch

# Prevent stale GPU state from crashing
torch.cuda.empty_cache() #clears cache help by pytorch to avoid CUDA crashes

# Ensure all CUDA asserts are reported clearly
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" #forces cuda operations to run in sync so errors are reported at the exact line where they occur instead of later
print("✅ CUDA debugging enabled") #confirmation to say debugging is active


✅ CUDA debugging enabled


In [12]:
import os

# Path to YOLO labels
base_label_dir = dataset_dir / "labels"                     #deifnes the base directory where yolo label files are stored (labels/train, labels/val, labels/test)

classes_found = set()                                       #create an empty set to store unique class IDs found in the dataset

for split in ["train", "val", "test"]:                      #loop through each dataset split
    split_dir = base_label_dir / split                      #constructs the full path to the labels directory for the current split

    if not split_dir.exists():                              #checks if split directory exists
        print(f"⚠️ Split folder missing: {split_dir}")      #prints a warning if the directory is missing
        continue                                            #skips to the next split

    for file in os.listdir(split_dir):                      #loop through all files in the current split directory
        if file.endswith(".txt"):                           #ensures only YOLO labels are processed
            with open(split_dir / file) as f:               #open the label file for reading
                for line in f:                              #iterates over each line in the label file
                    if line.strip():                        #cecks that the line is not empty or whitespace
                        cls = int(line.split()[0])          #extracts the class ID (first value in format)
                        classes_found.add(cls)              #adds the class IF to the set (duplicates ignored)
    
print("✅ Classes found in dataset:", sorted(classes_found))#prints sorted list of all unique class ids found acorss the dataset


✅ Classes found in dataset: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [13]:
from pathlib import Path

base = Path("datasets/VisDrone") #defines the base path pointing to the visdrone dataset directory

print("images/train:", len(list((base/"images/train").glob("*.jpg"))))  #count and print the number of training images by finding all .jpg files
print("images/val:",   len(list((base/"images/val").glob("*.jpg"))))    #count and print the number of validation images by finding all .jpg files
print("images/test:",  len(list((base/"images/test").glob("*.jpg"))))   #count and print the number of testing images by finding all .jpg files

print("labels/train:", len(list((base/"labels/train").glob("*.txt"))))  #count and print the number of training labels by finding all .txt files
print("labels/val:",   len(list((base/"labels/val").glob("*.txt"))))    #count and print the number of validation labels by finding all .txt files
print("labels/test:",  len(list((base/"labels/test").glob("*.txt"))))   #count and print the number of testing labels by finding all .txt files


images/train: 6471
images/val: 548
images/test: 1610
labels/train: 6471
labels/val: 548
labels/test: 1610


In [14]:
# ✅ Train YOLOv8 on VisDrone (detects vehicles and people)
from ultralytics import YOLO    #used to load and train YOLOv8 models

model = YOLO("yolov8n.pt")      #loads pretrained yolov8 nano model as a starting point (transfer learning to speed up training and improve convergance)
model.train(
    data=str(yaml_path),        #specifies path to dataset configuration file (data.yaml)
    epochs=10,                  #number of training cycles (full pass through the dataset)  
    imgsz=640,                  #sets the input image size for training (640x640 pixels)
    batch=8,                    #number of images proccessed per batch during training
    name="visdrone_parking_detector"    #name assigned to this training run (used for saving outputs in run/train/)
)


Ultralytics 8.4.11  Python-3.12.10 torch-2.11.0.dev20260203+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\VisDrone\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_parking_detector4, nbs=64, nms=False, opset=None, optimize=False, optimi

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002D1238F98E0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,   

In [15]:
# ✅ Evaluate model performance on validation set
model.val(data=str(yaml_path))


Ultralytics 8.4.11  Python-3.12.10 torch-2.11.0.dev20260203+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 914.8379.0 MB/s, size: 126.7 KB)
val: Scanning C:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\datasets\VisDrone\labels\val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 6.6it/s 5.3s0.1s
                   all        548      38759      0.354      0.273      0.255      0.146
            pedestrian        520       8844      0.337       0.31      0.272      0.113
                people        482       5125      0.393      0.223      0.213     0.0712
               bicycle        364       1287      0.166     0.0458     0.0409      0.013
                   car        515      14064      0.508      

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002D157642780>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,   

In [16]:
# ✅ Run inference on your parking lot video
video_path = "Assets/Video1.mp4"
output_dir = "C:/Users/these/FYP2025-SeanMaloney/YOLOv8/runs/detect"

results = model.predict(source=video_path, save=True, project="runs/detect", name="parking_output", show=False)
print("🎥 Inference complete. Output saved to:", results[0].save_dir)



WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/376) c:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\Assets\Video1.mp4: 384x640 5 pedestrians, 209 cars, 10 vans, 46.2ms
video 1/1 (frame 2/376) c:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\Assets\Video1.mp4: 384x640 3 pedestrians, 218 cars, 13 vans, 8.8ms
video 1/1 (frame 3/376) c:\Users\these\Desktop\FYP2025-SeanMaloney\YOLOv8\Assets\Video1.mp4: 384x640 2 pedestrians, 222 cars, 13 vans, 8.5ms
video 1/1 (frame 4/376) c:\Users\thes

Unused

In [17]:
base = "datasets/VisDrone"
val_images = os.listdir(f"{base}/images/val")
random.shuffle(val_images)
num_test = int(0.1 * len(val_images))
test_images = val_images[:num_test]

os.makedirs(f"{base}/images/test", exist_ok=True)
os.makedirs(f"{base}/labels/test", exist_ok=True)

for img in test_images:
    name = os.path.splitext(img)[0]
    shutil.move(f"{base}/images/val/{img}", f"{base}/images/test/{img}")
    if os.path.exists(f"{base}/labels/val/{name}.txt"):
        shutil.move(f"{base}/labels/val/{name}.txt", f"{base}/labels/test/{name}.txt")

print(f"✅ Created test set with {len(test_images)} images and labels.")


✅ Created test set with 54 images and labels.


In [18]:
for split in ["train", "val", "test"]:                                          #loops through each datatset split: training, valiation, test
    img_count = len(os.listdir(f"{dataset_dir}/images/{split}"))                #counts the number of image files in the current split's images directory
    label_count = len(os.listdir(f"{dataset_dir}/labels/{split}"))              #counts the number of labels in the current split's labels directory
    print(f"{split.capitalize()} → Images: {img_count} | Labels: {label_count}")#print formatted summary showing the number of images and labels for the current dataset split


Train → Images: 6471 | Labels: 6471
Val → Images: 494 | Labels: 494
Test → Images: 1664 | Labels: 1664


In [22]:
import cv2
import os

# Paths
input_path = "C:/Users/these/Desktop/FYP2025-SeanMaloney/runs/detect/runs/detect/parking_output/Video1.avi"    # change this if your AVI has a different name
output_path = "C:/Users/these/Desktop/FYP2025-SeanMaloney/runs/detect/runs/detect/parking_output/output.mp4"

# Check input exists
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Input file not found: {input_path}")

# Open video
cap = cv2.VideoCapture(input_path) #creates video capture object to read the input video file
if not cap.isOpened():  #checks if video was opened
    raise IOError("❌ Could not open input video.") #raise error if open cv fails to open the video

# Get properties
fps = cap.get(cv2.CAP_PROP_FPS) #retrives fps value of input video
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) #retrives width of each video in pixels
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))    #retrives height of each video in pixels
print(f"Video Info: {fps:.2f} FPS, {width}x{height}")   #print the basic video information for verification

# Create MP4 writer
fourcc = cv2.VideoWriter_fourcc(*'avc1')  # converts to Better compatibility than mp4v (H.264/AVC)
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))    #creates a videowriter object to write frames to the mp4 output file

# Convert frame by frame
frame_count = 0 #initalizes a counter to track how many frames are processed
while True: #infinite loop to read the video frame by frame
    ret, frame = cap.read() #reads next frame from input video, ret indicates whether the read was successful 
    if not ret: #if no more frame available
        break   #exit the loop
    out.write(frame)    #write the current frame to the output mp4 video
    frame_count += 1    #increment the frame counter

# Clean up
cap.release()   #release input video capture object
out.release()   #relkeases the output video writer and finalizes the mp4 file

print(f"✅ Conversion complete! {frame_count} frames saved to: {output_path}") #prints confirmation that the conversion finished successfully


Video Info: 24.00 FPS, 1920x1080
✅ Conversion complete! 372 frames saved to: C:/Users/these/Desktop/FYP2025-SeanMaloney/runs/detect/runs/detect/parking_output/output.mp4
